In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Create sinq

In [ ]:
sinq = Sinq(sinqname='dataset_1')
print(sinq.__repr__())

for id in sinq.df.index:
    sinq.add_tags(
        dayflycell=id,
        tags=['Figure1']
    )

sinq.display_df(show_tags=True)

In [ ]:
# sinq.restore_table(sinq.df.index[0])
# sinq.add_column('rest_time_fraction')
# sinq.add_column('drift_time_fraction')
# sinq.add_column('move_time_fraction')
# sinq.df.loc[sinq.df.index[0]]

In [ ]:
sinq.df.loc[sinq.df.index[0]]

In [ ]:
RECOMPUTE = False

# Standard compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
        # endofdaytrials = T.df.loc[1081:]
        # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
        sinq.add_table(table=T)
        sinq.df
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T


# Standard plotting
def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)

# Whole cell recordings

### 210302_F1_C2

In [ ]:
# # tr = Trial(fn='LEDFlashWithPiezoCueControl_Raw_210302_F1_C2_1.mat')
# # tr.params
# # tr = Trial(fn='LEDFlashWithPiezoCueControl_Raw_210302_F1_C2_1.mat')
# T = Table('LEDFlashWithPiezoCueControl_210302_F1_C2_Table.parquet',progress_bar=True)
# # T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# # fig,ax = T.plot_outcomes(savefig=True,format='png')
# # fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
id = '210302_F1_C2'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 210319_F2_C1 - Do not use
What is the deal!? There is an enormous number of points and the distributions is near probe_Zero, but the heatmap shows displacement. Not clear where this is coming from...?

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210319_F2_C1_Table.parquet',progress_bar=True,add_probestate=False)
# # Found 4 target positions - Counter({('lo', 730.0, -180.0, 60.0): 251, ('hi', 730.0, -260.0, 60.0): 199, ('no_state', 0.0, 520.0, 80.0): 50, ('no_state', 0.0, 450.0, 80.0): 50})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210319_F2_C1_1.mat')
# trial.fiberLED

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210331_F2_C1_Table.parquet',progress_bar=True,add_probestate=True)
# # T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# T.fig_folder = './figpanels'
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# # T.df.loc[775,'Trial'].exclude(reason='end_of_data_poorly_conditioned')


In [ ]:
id = '210331_F2_C1'
sinq.clear_note(dayflycell = id)
sinq.set_note(
        dayflycell = id,
        text= 'Seems like it should be learning',
        tags=['no_learn',],
        append=False,
)

sinq.display_df(show_tags=True)

# if id in sinq.df.index:
#       T = sinq.restore_table(dayflycell=id,overwrite=True)
# else:
#       raise KeyError(f'No index key {id}')
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)


In [ ]:
sinq.drop_tables()
sinq.display_df()

### 210405_F1_C1

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210405_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# # T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# T.fig_folder = './figpanels'
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# # sinq.df

id = '210405_F1_C1'
sinq.clear_note(dayflycell = id)
sinq.set_note(
        dayflycell = id,
        text= 'Long timeout, many timeouts',
        tags=['learn'],
        append=False,
)
# if RECOMPUTE:
#     sinq,T = run_compute(sinq,id)
#     run_plots(T)

### 210602_F1_C1 - Do not use
HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in, no attenuator on LED

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210602_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# Found 7 target positions - Counter({(490.0, 60.0): 241, (390.0, 60.0): 200, (310.0, 60.0): 101, (410.0, 60.0): 51, (380.0, 60.0): 50, (240.0, 60.0): 7, (140.0, 60.0): 1})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED
# T.df.loc[T.df.loc[:,'pyasXPosition']==310].index
# T.fig_folder = './figpanels'
# T.extract_trial_properties()
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
# fig
# T.fig_folder = './figpanels'
# print('Table: {}'.format(T))
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig

### 210604_F1_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210604_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# # Found 7 target positions - Counter({(490.0, 60.0): 241, (390.0, 60.0): 200, (310.0, 60.0): 101, (410.0, 60.0): 51, (380.0, 60.0): 50, (240.0, 60.0): 7, (140.0, 60.0): 1})
# # # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # # T.write_column_to_trial_files('pyasState')
# # # T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210604_F1_C1_1.mat')
# trial.fiberLED
# T.fig_folder = './figpanels'
# T.extract_trial_properties()
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
# fig
# T.fig_folder = './figpanels'
# print('Table: {}'.format(T))
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig

In [ ]:
id = '210604_F1_C1'
sinq.clear_note(dayflycell = id)
sinq.set_note(
        dayflycell = id,
        text= 'Long timeout, super learner',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 210903_F3_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_210903_F3_C1_Table.parquet',progress_bar=True,add_probestate=True)

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED
# T.fig_folder = './figpanels'
# T.extract_trial_properties()
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
# # fig
# T.fig_folder = './figpanels'
# print('Table: {}'.format(T))
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig

In [ ]:
id = '210903_F3_C1'
sinq.set_note(
        dayflycell = id,
        text= 'Long timeouts, ignores red, missing later blocks',
        tags=['learn'],
        append=False,
)
# sinq.display_df()

### 210915_F1_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_210915_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# Found 5 target positions - Counter({(432.0, 60.0): 208, (332.0, 60.0): 189, (2.0, 0.0): 50, (238.0, 60.0): 50, (338.0, 60.0): 50})

# T.exclude_list_of_trials(index = T.df.loc[T.df['pyasXPosition']==2,'Trial'].index,reason='unused target position')
# T.df.loc[T.df['pyasXPosition']==238,'Trial'].index
# T.df.loc[T.df['pyasXPosition']==238,'probeZero'] = 518
# T.df.loc[T.df['pyasXPosition']==338,'probeZero'] = 518
# T.df.loc[T.df['pyasXPosition']==338,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==238,'pyasState'] = 'hi'
# T.df.loc[T.df['pyasXPosition']==338,'pyasState']

# T.df.loc[T.df['pyasXPosition']==432.0,'probeZero'] = 612
# T.df.loc[T.df['pyasXPosition']==332.0,'probeZero'] = 612
# T.df.loc[T.df['pyasXPosition']==432.0,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==332.0,'pyasState'] = 'hi'
# T.df.loc[T.df['pyasXPosition']==338,'pyasState']
# T.write_column_to_trial_files('pyasState')
# T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED

In [ ]:
# index = T.df.loc[T.df['probeZero']==612.0].index

In [ ]:
# T.fig_folder = './figpanels'
# T.extract_trial_properties()
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[T.df['probeZero']==612.0].index,cmin=-500,cmax=10,format='png')
# fig

In [ ]:
# T.fig_folder = './figpanels'
# print('Table: {}'.format(T))
# # lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# # hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig

In [ ]:
# lo_fig

In [ ]:
id = '210915_F1_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 210917_F2_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_210917_F2_C1_Table.parquet',progress_bar=True,add_probestate=True)

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED

In [ ]:
# T.fig_folder = './figpanels'
# T.extract_trial_properties()
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
# fig

In [ ]:
# T.fig_folder = './figpanels'
# print('Table: {}'.format(T))
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig

In [ ]:
id = '210917_F2_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

# W+ Blue to Green

SS61350>w+;;pJFRC7 = +;pJFRC7/VT057379-p65ADZp in attP40; R20C08-ZpGdbd in attP2 (SS61350)

### 250226_F2_C1 - SS61350>w+;;pJFRC7

Nice fly_cell. There was the strange anticipation of the target change. There was a very long rest before switching to green LED. But even then 

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250226_F2_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=691,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=692,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    
id = '250226_F2_C1'
sinq.set_note(
        dayflycell = id,
        text= 'Never got it, tried hard',
        tags=['no_learn'],
        append=False,
)

# sinq.display_df()


# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)



### 250227_F1_C1 - SS61350>w+;;pJFRC7

Blue then Green. Really nice! Timeouts only begin with the green LED

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F1_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=510,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=511,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

### 250227_F2_C1 - SS61350>w+;;pJFRC7

Blue then Green. 
Nice learning! Some more timeouts and more as off trials with Green.

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F2_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=510,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=511,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

In [ ]:
id = '250227_F2_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250227_F3_C1 - SS61350>w+;;pJFRC7

?

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F3_C1_Table.parquet')

# T.add_df_category('blue',trial_min=1,trial_max=638,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=638,write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())
# T.open_notes_files()

In [ ]:
T.open_notes_files()

In [ ]:
id = '250227_F3_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250228_F1_C1 - SS61350>w+;;pJFRC7

Blue then Green. 
This one takes a long time to learn in each block. Some more timeouts and more as off trials with Green.

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250228_F1_C1_Table.parquet')
T.progress_bar = True
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=507,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=508,write_to_hdf5=True)
print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

In [ ]:
id = '250228_F1_C1'
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'Adjusted baseline, but never got it',
        tags=['no_learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250304_F3_C1 - SS61350>w+;;pJFRC7

Blue then Green. 


In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250304_F3_C1_Table.parquet')
# T.open_notes_files()
# T.extract_trial_properties()
# val = T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=1450,write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

In [ ]:
# fig,ax = T.plot_outcomes(savefig=True)
# # fname = T.export_some_probe_groups(index=idx, from_zero=True)
# # fig = plot_probe_trace_from_pickle(fname)
# fig.canvas.draw()
# figname =  f"./Figure1/example_outcomes_{T.day}_F{T.fly}_C{T.cell}_{T.genotype}_.svg"
# fig.savefig(fname=figname, format='svg')
# fig
T.extract_trial_properties()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '250304_F3_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250425_F1_C1 - +;pJFRC7;31H05-Gal4

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250425_F1_C1_Table.parquet')
# # T.assign_column_value('filtercube_status','blue',trial_min=1,write_to_hdf5=True)
# # T.open_notes_files()
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# print(T.df['fiberLED'].value_counts())

In [ ]:
id = '250425_F1_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250923_F1_C1 - +;pJFRC7;31H05-Gal4

In [ ]:
id = '250923_F1_C1'
# sinq.df.drop(index=id)

# if id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
#     T.extract_trial_properties()
#     T.find_successful_trials()
# else:
#     T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
#     T.exclude_trials()
#     T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     T.extract_trial_properties()
#     T.find_successful_trials()
#     sinq.add_table(table=T)
#     sinq.df

sinq.set_note(
        dayflycell = id,
        text= 'learns late?',
        tags=['no_learn'],
        append=False,
)
# sinq.display_df()
# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)


### 250923_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250923_F2_C1'
# sinq.df.drop(index=id)

# if id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
#     T.extract_trial_properties()
#     T.find_successful_trials()
# else:
#     T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
#     T.exclude_trials()
#     # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     T.extract_trial_properties()
#     T.find_successful_trials()
#     sinq.add_table(table=T)
#     sinq.df

# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '250923_F2_C1'
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 250924_F1_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250924_F1_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# T.find_successful_trials()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
        dayflycell = id,
        text= 'tires out',
        tags=['no_learn'],
        append=False,
)
# sinq.display_df()
# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)


### 250924_F2_C1 - SS61350>w+;;pJFRC7

Something funny happened here when I tried to exclude certain trials. I ended up excluding some I want to include. I suspect it has more to do with heatmap plotting

In [ ]:
id = '250924_F2_C1'
if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
# else:
#     T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
#     # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     # endofdaytrials = T.df.loc[1093:]
#     # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
#     sinq.add_table(table=T)
#     sinq.df

# T.extract_trial_properties()
# T.find_successful_trials()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# run_plots(T)
# sinq.set_note(
#         dayflycell = id,
#         text= 'na',
#         tags=['learn'],
#         append=False,
# )
# sinq.display_df(show_tags=True)



In [ ]:
fig = Figure(figsize = (8,6),dpi=200)
ax = fig.add_subplot(1,1,1)

tr1 = 521
tr2 = tr1+34
# tr1 = 804
# tr2 = tr1+27
fig,ax = T.plot_some_probe_groups(index=T.df.loc[tr1:tr2,:].index,fig=fig,ax=ax,force_pos=True)
display(fig)

In [ ]:
idx = T.df.index

mask = ~(
    ((idx >= 521) & (idx <= 553)) |
    ((idx >= 804) & (idx <= 831))
)

idx = idx[mask]

# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,index=idx,format='png')

df_filters = {'all':None,
            'hi':{'pyasState':'hi'},
            'lo':{'pyasState':'lo'},}

for filt_key in df_filters.keys():
    dff = df_filters[filt_key]
    fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                            df_filter=dff,index=idx,savefig=False,format='svg',
                            export=True)

### 250925_F1_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250925_F1_C1'
# if id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
# else:
#     T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
#     # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     # endofdaytrials = T.df.loc[1093:]
#     # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
#     sinq.add_table(table=T)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# sinq.add_table(table=T,overwrite=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'na',
        tags=['learn'],
        append=False,
)
# sinq.display_df()
# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)

### 250925_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250925_F2_C1'
# if id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
#     # endofdaytrials = T.df.loc[1081:]
#     # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
# else:
#     T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
#     T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     sinq.add_table(table=T)
#     sinq.df
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'tires out',
        tags=['no_learn'],
        append=False,
)
sinq.display_df(show_tags=True)

### 260521_F1_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '260521_F1_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# sinq.add_table(table=T)

# run_plots(T)
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'EMG in femur',
        tags=['learn'],
        append=False,
)

### 260521_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '260521_F2_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# sinq.add_table(table=T)
# sinq.df

# run_plots(T)
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'fly starts to learn at trial 500, EMG in femur',
        tags=['learn','late_learn','EMG in femur'],
        append=False,
)

sinq.display_df(show_tags=True)

# w+ Green to Blue

### 250228_F2_C1 - SS61350>w+;;pJFRC7

Green then Blue


In [ ]:
id = '250228_F2_C1'
# T = Table('LEDFlashTriggerPiezoControl_250228_F2_C1_Table.parquet')
# # T.add_df_category('green',trial_min=51,trial_max=713,categories='_filtercube_status_cat')
# # T = sinq.df.loc['250228_F2_C1','Table']
# # T.assign_column_value('filtercube_status','green',trial_min=51,trial_max=713,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# print(T.df['fiberLED'].value_counts())

sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'Strong learner',
        tags=['learn'],
        append=False,
)

# sinq.display_df()
# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)


### 250304_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250304_F2_C1_Table.parquet')
# T = sinq.df.loc['250304_F2_C1','Table']
# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=408,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','blue',trial_min=409,trial_max=513,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# print(T.df['fiberLED'].value_counts())

id = '250304_F2_C1'
sinq.clear_note(dayflycell = id)
sinq.set_note(
        dayflycell = id,
        text= 'green, frustrated',
        tags=['no_learn'],
        append=False,
)
sinq.display_df(show_tags=True)

# T = sinq.restore_table(dayflycell=id,overwrite=True)

# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)


### 250506_F2_C1 >- +;TH-Gal4;UAS-Kir2.1

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250506_F2_C1_Table.parquet')
# T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

# solution_ran_out = T.df.loc[494:]
# solution_ran_out.index
# T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
# sinq.add_table(table=T)
# sinq.df
# fig = Figure(figsize = (8,6),dpi=200)
# ax = fig.add_subplot(1,1,1)
# T.plot_some_probe_groups(T.df.loc[410:430,:].index,ax=ax,force_pos=True)
# fig


In [ ]:
id = '250506_F2_C1'
sinq.clear_note(dayflycell = id)
sinq.set_note(
        dayflycell = id,
        text= 'ok learner, got frustrated a bit, green',
        tags=['learn'],
        append=False,
)
# 

# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)

sinq.delete_table(id)
sinq.display_df(show_tags=True)

### 250514_F1_C1 >- +;TH-Gal4;UAS-Kir2.1

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250514_F1_C1_Table.parquet')
# # T.open_notes_files()

# # T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

# solution_ran_out = T.df.loc[494:]
# solution_ran_out.index
# T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
# sinq.add_table(table=T)
# sinq.df
# fig = Figure(figsize = (8,6),dpi=200)
# ax = fig.add_subplot(1,1,1)
# T.plot_some_probe_groups(T.df.loc[410:430,:].index,ax=ax,force_pos=True)
# fig

In [ ]:
# id = '250514_F1_C1'
# # sinq.set_note(
# #         dayflycell = id,
# #         text= 'OK learner, timed out a bit',
# #         tags=['learn'],
# #         append=False,
# # )

# # sinq.display_df()

# # T = sinq.restore_table(dayflycell=id,overwrite=True)
# # T.exclude_trials()
# # T.extract_trial_properties()
# # T.find_successful_trials()
# # run_plots(T)

# sinq.delete_table(id)